# Mini Semantic Search

## 1. Project Overview

This project builds a mini semantic search system using customer support question-and-answer data.

Unlike traditional keyword search, semantic search aims to find relevant results based on the meaning of a query, even when the wording differs from the stored text.

We will use Pandas and NumPy for data preparation, a text embedding model to represent text as numerical vectors, and ChromaDB to store and retrieve relevant results.


## 2. Load the Dataset

We will use Pandas' `read_csv()` function to load our customer support dataset into a DataFrame. A DataFrame organizes the data into rows and columns, allowing us to inspect, clean, and prepare the questions and answers for semantic search.

In [76]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")

print(df.shape)
print(df.head())

(26872, 5)
   flags                                        instruction category  \
0      B   question about cancelling order {{Order Number}}    ORDER   
1    BQZ  i have a question about cancelling oorder {{Or...    ORDER   
2   BLQZ    i need help cancelling puchase {{Order Number}}    ORDER   
3     BL         I need to cancel purchase {{Order Number}}    ORDER   
4  BCELN  I cannot afford this order, cancel purchase {{...    ORDER   

         intent                                           response  
0  cancel_order  I've understood you have a question regarding ...  
1  cancel_order  I've been informed that you have a question ab...  
2  cancel_order  I can sense that you're seeking assistance wit...  
3  cancel_order  I understood that you need assistance with can...  
4  cancel_order  I'm sensitive to the fact that you're facing f...  


### 2.2 Understanding the Dataset

Our dataset contains customer support requests and their corresponding responses. Before cleaning the data, we will examine its columns, data types, and overall structure.

Understanding these details will help us identify which columns are useful for semantic search and whether any data needs to be cleaned.

In [77]:
print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Dataset dimensions: (26872, 5)

Column names:
['flags', 'instruction', 'category', 'intent', 'response']

Data types:
flags          str
instruction    str
category       str
intent         str
response       str
dtype: object

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   flags        26872 non-null  str  
 1   instruction  26872 non-null  str  
 2   category     26872 non-null  str  
 3   intent       26872 non-null  str  
 4   response     26872 non-null  str  
dtypes: str(5)
memory usage: 1.0 MB


### 2.3 Checking Data Quality

Before preparing our text for semantic search, we need to check for duplicate records, empty values, and whitespace-only text.

We will use Pandas to identify these issues and NumPy to help summarize the results. We will inspect the data first rather than immediately deleting anything.

In [78]:
# Count duplicate rows
duplicate_count = df.duplicated().sum()

# Count missing values in each column
missing_values = df.isna().sum()

# Count empty or whitespace-only strings in each column
blank_values = df.apply(
    lambda column: column.str.strip().eq("").sum()
)

print("Duplicate rows:", duplicate_count)

print("\nMissing values per column:")
print(missing_values)

print("\nBlank strings per column:")
print(blank_values)

Duplicate rows: 0

Missing values per column:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Blank strings per column:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64


### 2.4 Selecting Relevant Columns

Our goal is to find customer support answers that are relevant to a user's question. The `instruction` column contains customer requests, while the `response` column contains the corresponding answers.

We will create a function called `prepare_data()` to select these columns and rename them to `question` and `answer`. Keeping this operation inside a function makes it reusable and keeps our notebook organized.

In [79]:
def prepare_data(df):
    # Select the columns needed for semantic search
    faq_df = df[["instruction", "response"]].copy()

    # Rename the columns
    faq_df = faq_df.rename(
        columns={
            "instruction": "question",
            "response": "answer"
        }
    )

    return faq_df


faq_df = prepare_data(df)

print("Prepared dataset shape:", faq_df.shape)
display(faq_df.head())

Prepared dataset shape: (26872, 2)


,question,answer
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...


### 2.5 Text Preprocessing

Text preprocessing helps make our dataset more consistent before semantic search. We will define a `clean_text()` function that converts text to lowercase and removes unnecessary leading, trailing, and repeated whitespace.

We will preserve punctuation and the original wording because they may carry useful meaning. We will then apply the function to both questions and answers.

In [80]:
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove unnecessary whitespace
    text = " ".join(text.split())

    return text


# Clean questions and answers
faq_df["question"] = faq_df["question"].apply(clean_text)
faq_df["answer"] = faq_df["answer"].apply(clean_text)

display(faq_df.head())

,question,answer
0,question about cancelling order {{order number}},i've understood you have a question regarding ...
1,i have a question about cancelling oorder {{or...,i've been informed that you have a question ab...
2,i need help cancelling puchase {{order number}},i can sense that you're seeking assistance wit...
3,i need to cancel purchase {{order number}},i understood that you need assistance with can...
4,"i cannot afford this order, cancel purchase {{...",i'm sensitive to the fact that you're facing f...


### 2.6 Cleaning and Validating the Dataset

Even after normalizing text, we should ensure that every record contains both a question and an answer. We will define a `clean_dataset()` function to remove missing values, empty text, and exact duplicate question-answer pairs.

The function will return a cleaned DataFrame and report how many records were removed. This helps us track the effect of our cleaning process.

In [81]:
def clean_dataset(df):
    original_count = len(df)

    # Remove rows with missing questions or answers
    df = df.dropna(subset=["question", "answer"]).copy()

    # Remove rows where either field is empty
    df = df[
        df["question"].str.strip().ne("")
        & df["answer"].str.strip().ne("")
    ].copy()

    # Remove exact duplicate question-answer pairs
    df = df.drop_duplicates(subset=["question", "answer"])

    removed_count = original_count - len(df)

    print("Original records:", original_count)
    print("Records removed:", removed_count)
    print("Remaining records:", len(df))

    return df.reset_index(drop=True)


faq_df = clean_dataset(faq_df)

Original records: 26872
Records removed: 0
Remaining records: 26872


## 3. Generating Text Embeddings

Text embedding models convert text into numerical vectors that represent semantic meaning. These vectors allow our search system to compare questions based on meaning rather than relying only on exact keyword matches.

We will use the pretrained `all-MiniLM-L6-v2` model from Sentence Transformers. It is a lightweight model suitable for a small semantic search project.

### 3.1 Loading the Model

We will load the pretrained model and use it in the next step to convert our customer support questions into embeddings. The model may need to download its files the first time it runs.

In [82]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


### 3.2 Creating Text Embeddings

The embedding model converts text into numerical vectors that capture aspects of its meaning. Text with similar meanings can have similar vector representations, even when the wording differs.

We will create a reusable `create_embeddings()` function that accepts a list of texts and returns their embeddings. First, we will test it with a few example questions before processing the full dataset.

In [83]:
def create_embeddings(texts):
    embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    return embeddings


# Test the function with three example questions
sample_questions = [
    "How can I cancel my order?",
    "I want to return a product.",
    "Where is my package?"
]

sample_embeddings = create_embeddings(sample_questions)

print("Embedding shape:", sample_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (3, 384)


### 3.3 Generating Embeddings for the Dataset

Our test confirmed that the embedding model works. We will now use `create_embeddings()` to convert all customer support questions into numerical vectors.

Each question will have a 384-dimensional embedding. We will store the resulting NumPy array in `question_embeddings` for use in our semantic search system.

In [84]:
# Generate embeddings for all customer support questions
questions = faq_df["question"].tolist()

question_embeddings = create_embeddings(questions)

print("Number of questions:", len(questions))
print("Embeddings shape:", question_embeddings.shape)

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

Number of questions: 26872
Embeddings shape: (26872, 384)


## 4. Storing Embeddings with ChromaDB

We have generated embeddings for all 26,872 customer support questions. Next, we will use ChromaDB to store these vectors alongside their corresponding questions and answers.

ChromaDB organizes stored records into collections. We will create a collection for our customer support data and use it to retrieve relevant answers when a user submits a query.

In [85]:
import chromadb

# Create a ChromaDB client that stores data locally
client = chromadb.PersistentClient(path="../chroma_db")

# Create or retrieve our collection
collection = client.get_or_create_collection(
    name="customer_support_faqs",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection is ready!")

ChromaDB collection is ready!


### 4.2 Storing Questions, Answers, and Embeddings

We will create a reusable `store_documents()` function to save our customer support data in ChromaDB.

Each record will have a unique ID, a question as its document, an answer stored as metadata, and its corresponding embedding. We will insert the records in batches to manage the dataset efficiently.

Because our ChromaDB collection uses persistent storage, the saved records will remain available between notebook sessions.

In [86]:
def store_documents(collection, faq_df, embeddings, batch_size=500):
    total_records = len(faq_df)

    for start in range(0, total_records, batch_size):
        end = min(start + batch_size, total_records)

        # Prepare one batch of records
        batch_df = faq_df.iloc[start:end]

        collection.upsert(
            ids=[str(i) for i in range(start, end)],
            documents=batch_df["question"].tolist(),
            metadatas=[
                {"answer": answer}
                for answer in batch_df["answer"].tolist()
            ],
            embeddings=embeddings[start:end].tolist()
        )

    print(f"Stored {total_records} records in ChromaDB.")


store_documents(collection, faq_df, question_embeddings)

Stored 26872 records in ChromaDB.


## 5. Searching for Relevant Answers

Our customer support questions and their embeddings are now stored in ChromaDB. We can use this collection to find answers relevant to a user's query.

When a user submits a question, we will generate an embedding for it and compare it with the stored embeddings using ChromaDB. The system will return the three most relevant questions and their corresponding answers.

In [ ]:
def search_faqs(query, collection, model, top_k=3):
    # Convert the user's query into an embedding
    query_embedding = model.encode(
        query,
        convert_to_numpy=True
    )

    # Find the most relevant stored questions
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    # Display the results
    for i in range(len(results["ids"][0])):
        question = results["documents"][0][i]
        answer = results["metadatas"][0][i]["answer"]
        distance = results["distances"][0][i]

        print(f"\nResult {i + 1}")
        print(f"Question: {question}")
        print(f"Answer: {answer}")
        print(f"Cosine distance: {distance:.4f}")
        print("-" * 50)

### 5.2 Testing Semantic Search

We will test our search function with a customer support question about cancelling an order. We will inspect the top three results to see whether the system retrieves relevant questions and answers, even when their wording differs from the query.

In [90]:
print("Starting search...")

search_faqs(
    query="Where is my package?",
    collection=collection,
    model=model
)

print("Search finished.")

Starting search...

Result 1
Question: i cannot pay for the last purchase i have made, cancel it
Answer: i gather that you're unable to make the payment for your recent purchase and would like to cancel it. i apologize for any inconvenience this may have caused. to assist you in canceling the purchase and resolving this issue, please follow the steps below: 1. log into your {{online company portal info}} using your credentials. 2. locate the '{{online order interaction}}' or '{{online order interaction}}' section, and navigate to the specific purchase you would like to cancel. 3. select the option labeled '{{online order interaction}}' associated with your purchase. 4. take note that there may be additional instructions or questions prompted by the system, and your cooperation in providing the necessary information is greatly appreciated. if you encounter any difficulties or have further questions, our dedicated support team is available to assist you during {{customer support hours}}.